In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score

# =====================================================================
# [테스트용] 실제 titanic.csv가 없어도 작동하도록 가상 데이터 생성 (실습 시 삭제 가능)
# =====================================================================
np.random.seed(42)
mock_data = {
    'Pclass': np.random.choice([1, 2, 3], size=200),
    'Age': np.random.choice([22.0, 38.0, 26.0, np.nan], size=200),
    'SibSp': np.random.randint(0, 3, size=200),
    'Parch': np.random.randint(0, 3, size=200),
    'Fare': np.random.exponential(scale=30, size=200) + 10,
    'Gender': np.random.choice(['male', 'female'], size=200),
    'Embarked': np.random.choice(['S', 'C', 'Q', np.nan], size=200),
    'Cabin': np.random.choice(['C123', np.nan], size=200),
    'Survived': np.random.choice([0, 1], size=200)
}
import os; os.makedirs('datasets', exist_ok=True)
pd.DataFrame(mock_data).to_csv('datasets/titanic.csv', index=False)
# =====================================================================

# ─── 1단계: 데이터 로드 및 기본 탐색 ─────────────────────────
df = pd.read_csv('datasets/titanic.csv')
print('Shape:', df.shape)
print(df.info())
print(df.describe())
print('-' * 60)

# ─── 2단계: 결측값 현황 확인 및 1차 정제 ─────────────────────────
print('결측값 현황:')
print(df.isnull().sum())
print('-' * 60)

# [보완] Cabin 제거와 같은 구조적 변형은 미리 수행하되, 
# 데이터 누수를 유발하는 'Age' 평균값과 'Embarked' 최빈값 채우기는 아래 6단계 파이프라인 내부로 이관합니다.
df.drop('Cabin', axis=1, inplace=True)

# ─── 3단계: 이상치 탐지 (Fare 기준 3σ) ───────────────────────
fare_mean = df['Fare'].mean()
fare_std  = df['Fare'].std()

# [보완] SettingWithCopyWarning 방지를 위해 필터링 후 .copy()를 명시합니다.
df = df[abs(df['Fare'] - fare_mean) <= 3 * fare_std].copy()
print('이상치 제거 후 Shape:', df.shape)
print('-' * 60)

# ─── 4단계 & 5단계: 특성 선택 및 데이터 분리 ──────────────────────────
# [보완] 원핫 인코딩과 라벨 인코딩을 데이터프레임에 미리 적용하면 교차 검증 시 누수가 발생합니다.
# 원본 형태의 특성들을 선택하고, 인코딩 공정은 파이프라인이 담당하도록 양보합니다.
features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Gender', 'Embarked']
X = df[features]
y = df['Survived']

# ─── 6단계: Pipeline 및 ColumnTransformer 구성 ──────────────────────────────────────
# [보완] 데이터 타입별로 전처리 공정(결측치 처리 + 인코딩/스케일링)을 완벽하게 분리합니다.

# ① 수치형 변수 공정: 나이 결측치는 평균('mean')으로 채우고 표준화 스케일링
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']
numeric_transformer = make_pipeline(
    SimpleImputer(strategy='mean'),
    StandardScaler()
)

# ② 범주형 변수 공정: 성별/탑승지는 최빈값('most_frequent') 채우고 원핫 인코딩
# (Gender는 이진 값이므로 원핫 인코딩을 적용하면 자연스럽게 수치화되며, LabelEncoder의 한계를 극복합니다.)
categorical_features = ['Gender', 'Embarked']
categorical_transformer = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=int)
)

# ③ 개별 공정을 하나로 묶는 컴포저(Preprocessor) 생성
# 'Pclass'는 변환 없이 그대로 모델에 전달(passthrough)합니다.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

# ④ 전처리 컴포저와 모델을 하나의 최종 파이프라인으로 병합
pipeline = make_pipeline(
    preprocessor,
    LogisticRegression(random_state=42, max_iter=2000)
)

# ─── 7단계: Stratified K-Fold 교차 검증 (5-Fold) ──────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_val_score(
    pipeline, X, y,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1
)

print('Cross Validation Results:')
print(f'  각 Fold 정확도: {cv_results}')
print(f'  평균 정확도:   {cv_results.mean():.4f}')
print(f'  표준 편차:     {cv_results.std():.4f}')

Shape: (200, 9)
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    200 non-null    int64  
 1   Age       135 non-null    float64
 2   SibSp     200 non-null    int64  
 3   Parch     200 non-null    int64  
 4   Fare      200 non-null    float64
 5   Gender    200 non-null    str    
 6   Embarked  157 non-null    str    
 7   Cabin     102 non-null    str    
 8   Survived  200 non-null    int64  
dtypes: float64(2), int64(4), str(3)
memory usage: 14.2 KB
None
           Pclass         Age       SibSp       Parch        Fare    Survived
count  200.000000  135.000000  200.000000  200.000000  200.000000  200.000000
mean     2.035000   28.459259    0.940000    0.870000   38.282367    0.460000
std      0.835022    6.753399    0.799749    0.834615   33.095789    0.499648
min      1.000000   22.000000    0.000000    0.000000   10.439544    0.000000
25%     